# PTCG Agent — Revised Implementation Plan (Kaggle Notebook)

**Status: source of truth. Replaces all previous plans.**

This notebook is the executable counterpart to the Jul 26 revised plan. Every markdown
section below corresponds 1:1 to a clause in that plan; every code cell either implements
that clause or provides a runnable scaffold for it (with clearly marked `TODO`s where the
real competition harness — `cg.api`, `search_begin`/`search_step` — must be wired in on
Kaggle, since it isn't importable in a generic dev sandbox).

> **Methodology note (added this revision):** a prior literature-review pass for this
> project contained fabricated and misattributed citations (a nonexistent "TradingStone"
> paper; real-paper citations with invented findings bolted on). Nothing in this notebook
> depends on that review. Every technique used below is either (a) elementary/derivable
> math (BCE gradient, Wilson interval), or (b) a specific, checkable, real source cited
> inline (AlphaGo Zero's 55% gating threshold — Silver et al. 2017, *Nature*; Wilson score
> interval — Wilson 1927, *JASA*; group-wise splitting — `sklearn.model_selection.GroupShuffleSplit`).


## Competition Structure (Two Tracks, One Agent)

| Track | Deliverable | Deadline | Weight | Status |
|-------|-------------|----------|--------|--------|
| **Simulation** | `submission.tar.gz` (`main.py` + `deck.csv` + `cg/`) | Aug 16 23:59 UTC | Skill rating via TrueSkill/Elo | ✅ Baseline submitted (heuristic-only, ~664 tier) |
| **Strategy** | 2,000-word report analyzing the agent's design, decisions, reasoning | Sept 6 entry / Sept 13 final | Judged by humans; **gatekeeper to $240K Tokyo finals** | ❌ Does not exist |

The Strategy report needs **real ablation data** from the Simulation agent. It can't start
until the agent stabilizes (Phase C). This is a **hard dependency chain**, not two parallel
tracks — this notebook is sequenced accordingly (Phase A → B → C → Report).


## Current Agent — Honest Assessment

`main.py` is a **pure heuristic scoring engine**. Every decision comes from hand-crafted
integer scores in `score_option()`. The neural scaffolding (`StateEncoder`, `NeuralWorker`)
is structurally present but **completely inert** (returns zeros / random weights / never
called).

Two heuristic engines with similar priority orderings converge to similar ratings — our
agent is **capped near the sample agent's ~664 tier by construction**. Breaking past that
ceiling requires activating the neural components (Phase B), which requires training data,
which requires real captured observations (Phase A).

### Unverified Assumptions (High Risk)
- `attackId` as a local index — assumed, never confirmed against a real game observation
- `energies` field format — assumed to be raw ints, never inspected from wire data
- No real observation has been captured and manually inspected despite `GameLogger` being wired

### Missing Components
- No non-regression gate (can submit a worse agent while believing it's better)
- No deck construction rationale (Deck Score = 20% of Strategy grade)
- No matchup-spread testing
- Strategy report: 0 words written (10% of score, gate to finalist tier)


In [ ]:
# Environment setup
# Everything below is written to run standalone (mock fallbacks) so this notebook
# executes cleanly even before `cg` is available on the Kaggle harness. Swap the
# `USE_MOCK_ENGINE` flag once the real package is importable.

import json
import math
import random
import shutil
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

USE_MOCK_ENGINE = True

try:
    import cg  # noqa: F401  (real competition package, only present on Kaggle harness)
    USE_MOCK_ENGINE = False
except ImportError:
    pass

print(f"USE_MOCK_ENGINE = {USE_MOCK_ENGINE}")


## Phase A: Close the Verification Loop (Now → Aug 2)

**Goal**: De-risk everything that follows. **Zero rating improvement expected** from this
phase — it produces facts, not points.

### Task 1 — Capture a real observation

Run one actual game (or use Kaggle's validation step), capture the raw JSON via
`GameLogger`, and manually inspect it to settle:
- Does `attackId` index locally into the active Pokémon's attack list?
- Does `energies` arrive as raw ints or as objects?
- What fields are actually present vs. assumed?

**This is the single highest-leverage unclosed task in the whole plan** — Phase B cannot
be trusted until this cell has been run against a real observation at least once.


In [ ]:
# Phase A / Task 1: capture + manually inspect one real observation
# On Kaggle: replace `mock_observation()` with the actual observation object the
# harness passes into your agent's `act()`/`step()` entrypoint, run one real game,
# and inspect the printed structure by hand against the three open questions above.

def mock_observation():
    """Stand-in shape ONLY — do not trust these field names/types until Task 1
    has been run against a real Kaggle observation and this function's assumptions
    have been corrected or confirmed."""
    return {
        "active": {"hp": 180, "maxHp": 220, "energies": [1, 1, 1, 0], "attacks": [
            {"attackId": 0, "name": "Aura Sphere", "cost": [1, 1]},
            {"attackId": 1, "name": "Power Blast", "cost": [1, 1, 1, 1]},
        ]},
        "bench": [],
        "opponent": {"active": {"hp": 130, "maxHp": 200}, "bench": [], "prizes": 4},
        "hand": [],
        "deckCount": 34,
        "prizes": 5,
    }


def capture_and_inspect(observation, log_path="game_log.jsonl"):
    """Append the raw observation to GameLogger-style JSONL and print a field
    inventory so the three open assumptions can be checked by eye."""
    record = {"ts": time.time(), "observation": observation}
    with open(log_path, "a") as f:
        f.write(json.dumps(record) + "\n")

    print("--- Field inventory (Phase A / Task 1 checklist) ---")
    print(f"attackId present per attack: {[('attackId' in a) for a in observation['active']['attacks']]}")
    print(f"attackId values (check: local index into attack list?): "
          f"{[a['attackId'] for a in observation['active']['attacks']]}")
    print(f"energies type (check: raw ints or objects?): {type(observation['active']['energies'][0]).__name__}")
    print(f"top-level keys present: {sorted(observation.keys())}")
    return record

obs = mock_observation()
_ = capture_and_inspect(obs)


### Task 2 — Populate `StateEncoder`

Fill the 84-dim vector with **real** features (not zeros):

| Block | Dims |
|---|---|
| My/Opp active HP ratio | 2 floats |
| My/Opp bench count, hand count, deck count | 6 ints (normalized) |
| My/Opp prize count + prize differential | 3 floats |
| Energy counts by type on active | 6 floats |
| Active Pokémon type matchup — weakness/resistance | 4 bools |
| Board threat level from AttackPlan (damage ratio, KO feasibility) | 4 floats |
| Remaining slots: bench HP ratios, hand composition summary | fills to 84 |


In [ ]:
# Phase A / Task 2: StateEncoder — real implementation, indexed and auditable
# (replaces the previous `np.zeros(84)` stub)

STATE_DIM = 84
ENERGY_TYPES = ["fighting", "colorless", "psychic", "fire", "water", "lightning"]

def _safe_ratio(hp, max_hp):
    return 0.0 if not max_hp else max(0.0, min(1.0, hp / max_hp))

def build_state_vector(observation) -> np.ndarray:
    """Populate the 84-dim feature vector from a (real or mock) observation.
    Every slice below is explicitly indexed and commented so it is auditable --
    unlike a black-box encoder, a reviewer can trace exactly which float came
    from which game fact.
    """
    v = np.zeros(STATE_DIM, dtype=np.float32)
    i = 0

    me = observation["active"]
    opp = observation["opponent"]["active"]

    # --- HP ratios (2) ---
    v[i] = _safe_ratio(me["hp"], me["maxHp"]); i += 1
    v[i] = _safe_ratio(opp["hp"], opp["maxHp"]); i += 1

    # --- bench / hand / deck counts, normalized, mine + opponent's (6) ---
    v[i] = min(len(observation.get("bench", [])), 5) / 5.0; i += 1
    v[i] = min(len(observation.get("hand", [])), 10) / 10.0; i += 1
    v[i] = min(observation.get("deckCount", 0), 60) / 60.0; i += 1
    v[i] = min(len(observation["opponent"].get("bench", [])), 5) / 5.0; i += 1
    v[i] = 0.0; i += 1  # opponent hand size is hidden info -- placeholder until Stage 2 (belief state)
    v[i] = 0.0; i += 1  # opponent deck count -- fill in once field is confirmed in Task 1

    # --- prize counts + differential (3) ---
    my_prizes = observation.get("prizes", 6)
    opp_prizes = observation["opponent"].get("prizes", 6)
    v[i] = my_prizes / 6.0; i += 1
    v[i] = opp_prizes / 6.0; i += 1
    v[i] = (opp_prizes - my_prizes) / 6.0; i += 1  # positive = I'm ahead on prize race

    # --- energy counts by type on active (6) ---
    energies = me.get("energies", [])
    for t in range(len(ENERGY_TYPES)):
        v[i] = (energies[t] if t < len(energies) else 0) / 4.0; i += 1

    # --- weakness / resistance booleans (4) -- placeholders until card DB lookup is wired ---
    v[i] = 0.0; i += 1  # I'm weak to opponent's active type
    v[i] = 0.0; i += 1  # I resist opponent's active type
    v[i] = 0.0; i += 1  # opponent is weak to my active type
    v[i] = 0.0; i += 1  # opponent resists my active type

    # --- board threat level from AttackPlan (4) -- placeholders, wire to AttackPlan output ---
    v[i] = 0.0; i += 1  # best available damage / opp remaining HP
    v[i] = 0.0; i += 1  # KO feasible this turn (0/1)
    v[i] = 0.0; i += 1  # opponent's best damage / my remaining HP (threat to me)
    v[i] = 0.0; i += 1  # opponent KO feasible against me (0/1)

    # --- remaining slots: bench HP ratios (5) + hand composition summary, pad to 84 ---
    for _ in range(5):
        v[i] = 0.0; i += 1  # bench slot HP ratio, fill once bench Pokemon HP is inspected

    # pad remaining dims explicitly to 84 (keeps the vector auditable -- no silent zeros)
    while i < STATE_DIM:
        v[i] = 0.0; i += 1

    assert i == STATE_DIM
    return v


def validate_state_vector(v: np.ndarray):
    assert v.shape == (STATE_DIM,), f"expected shape ({STATE_DIM},), got {v.shape}"
    assert not np.isnan(v).any(), "state vector contains NaNs"
    assert np.all((v >= -1.0) & (v <= 1.0)), "state vector has out-of-range values"
    return True

sv = build_state_vector(obs)
validate_state_vector(sv)
print(f"state vector shape={sv.shape}, non-zero dims={int(np.count_nonzero(sv))}/{STATE_DIM}")


### Task 3 — Improve heuristic scoring (quick wins only)

- Item-specific play weights (Ultra Ball > Potion > Switch)
- Energy type matching (attach Fighting to Lucario, not random)
- Bench development priority


In [ ]:
# Phase A / Task 3: quick heuristic wins to merge into score_option()

ITEM_WEIGHTS = {
    "Ultra Ball": 5700,   # search power > generic item baseline (5500)
    "Nest Ball":  5650,
    "Switch":     5550,
    "Potion":     5500,   # baseline item score from the existing engine
}

def energy_type_match_bonus(attacker_type: str, energy_type: str) -> int:
    """Small additive bonus so the scorer prefers attaching the attacker's own
    type over a mismatched Colorless-only energy, all else equal."""
    return 150 if attacker_type == energy_type else 0

def bench_priority_bonus(bench_count: int, hand_has_basics: bool) -> int:
    """Nudge bench development earlier when the bench is thin and a basic is
    available, tapering off as the bench fills toward 5."""
    if not hand_has_basics:
        return 0
    return max(0, (3 - bench_count)) * 100  # +300 at 0 benched, +0 at 3+

print(ITEM_WEIGHTS)
print("energy_type_match_bonus('fighting','fighting') =", energy_type_match_bonus("fighting", "fighting"))
print("bench_priority_bonus(1, True) =", bench_priority_bonus(1, True))


## Phase B: The Only Phase That Moves Rating (Aug 2 → Aug 9)

**Goal**: Break past the ~664 heuristic ceiling.

### B1. Supervised Value Network Training

**Loss** (binary cross-entropy on game outcome):

$$L = -[y \log \hat{y} + (1-y) \log(1-\hat{y})]$$

where $y$ = game outcome (1 win, 0 loss; drops or 0.5 for draws — documented below).

**Gradient at the sigmoid output** (hand-derived, no autograd):

$$\frac{\partial L}{\partial z} = \hat{y} - y$$

backpropagated through ReLU layers via a `> 0` mask.

**Data split**: split by **entire game**, not by (state, action) pair. Decisions within
one game are correlated (same deck, same opponent, same trajectory) — shuffling at the
decision level leaks information and gives optimistic validation scores. **80/20
game-level split** (equivalent to `sklearn.model_selection.GroupShuffleSplit` with
`groups=game_id`).

**Hyperparameters** (for ~7.5K params, small dataset):
- Learning rate: $10^{-3}$ to $10^{-2}$, plain SGD or SGD+momentum
- Mini-batch: 32–64 (full-batch OK below a few hundred games)
- L2 weight decay: $10^{-4}$ to $10^{-3}$ (self-play data is small and correlated)
- Early stopping on **validation win-rate**, not validation loss


In [ ]:
# Phase B / B1a: game-level 80/20 split (no leakage)

def load_game_log(path="game_log.jsonl"):
    """Each line is one (observation, action, ...) record tagged with a game_id.
    On Kaggle, GameLogger should stamp every record with the game_id it belongs to;
    add that stamp now if it isn't already there -- it's required for a correct split.
    """
    records = []
    p = Path(path)
    if not p.exists():
        return records
    with open(p) as f:
        for line in f:
            records.append(json.loads(line))
    return records


def group_train_val_split(records, val_frac=0.2, seed=0):
    """Group-wise split by game_id -- mirrors sklearn.model_selection.GroupShuffleSplit.
    Splitting at the decision level instead of the game level is a data-leakage bug:
    two decisions from the same game share the deck, the opponent, and the trajectory,
    so a decision-level shuffle lets the model 'peek' at correlated future/past states.
    """
    game_ids = sorted({r.get("game_id", "unassigned") for r in records})
    rng = random.Random(seed)
    rng.shuffle(game_ids)
    n_val_games = max(1, int(len(game_ids) * val_frac)) if game_ids else 0
    val_ids = set(game_ids[:n_val_games])
    train = [r for r in records if r.get("game_id", "unassigned") not in val_ids]
    val = [r for r in records if r.get("game_id", "unassigned") in val_ids]
    return train, val

records = load_game_log()
train_records, val_records = group_train_val_split(records)
print(f"loaded {len(records)} records -> train={len(train_records)} val={len(val_records)}")
if not records:
    print("No game_log.jsonl found yet -- this cell will populate once Phase A "
          "self-play logging is running. Training cells below use synthetic data "
          "as a smoke test in the meantime.")


In [ ]:
# Phase B / B1b: tiny value network, numpy-only (matches the "no autograd" spec)
# Architecture: 84 -> 64 -> 32 -> 1 (sigmoid), ReLU hidden activations.

class ValueNet:
    def __init__(self, in_dim=84, h1=64, h2=32, seed=0):
        rng = np.random.default_rng(seed)
        # He init for ReLU layers, small init for output
        self.W1 = rng.normal(0, np.sqrt(2 / in_dim), size=(in_dim, h1)).astype(np.float32)
        self.b1 = np.zeros(h1, dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2 / h1), size=(h1, h2)).astype(np.float32)
        self.b2 = np.zeros(h2, dtype=np.float32)
        self.W3 = rng.normal(0, np.sqrt(2 / h2), size=(h2, 1)).astype(np.float32)
        self.b3 = np.zeros(1, dtype=np.float32)
        # momentum buffers
        self._m = {k: np.zeros_like(v) for k, v in self.params().items()}

    def params(self):
        return {"W1": self.W1, "b1": self.b1, "W2": self.W2, "b2": self.b2,
                "W3": self.W3, "b3": self.b3}

    def n_params(self):
        return sum(p.size for p in self.params().values())

    def forward(self, X):
        z1 = X @ self.W1 + self.b1
        a1 = np.maximum(0, z1)
        z2 = a1 @ self.W2 + self.b2
        a2 = np.maximum(0, z2)
        z3 = a2 @ self.W3 + self.b3
        yhat = 1 / (1 + np.exp(-z3))
        cache = (X, z1, a1, z2, a2, z3, yhat)
        return yhat, cache

    def backward(self, cache, y, l2=1e-4):
        X, z1, a1, z2, a2, z3, yhat = cache
        n = X.shape[0]
        y = y.reshape(-1, 1)

        # dL/dz3 = yhat - y  (BCE + sigmoid combined gradient)
        dz3 = (yhat - y) / n
        dW3 = a2.T @ dz3 + l2 * self.W3
        db3 = dz3.sum(axis=0)

        da2 = dz3 @ self.W3.T
        dz2 = da2 * (z2 > 0)  # ReLU mask
        dW2 = a1.T @ dz2 + l2 * self.W2
        db2 = dz2.sum(axis=0)

        da1 = dz2 @ self.W2.T
        dz1 = da1 * (z1 > 0)
        dW1 = X.T @ dz1 + l2 * self.W1
        db1 = dz1.sum(axis=0)

        return {"W1": dW1, "b1": db1, "W2": dW2, "b2": db2, "W3": dW3, "b3": db3}

    def step(self, grads, lr=1e-3, momentum=0.9):
        for k, p in self.params().items():
            self._m[k] = momentum * self._m[k] - lr * grads[k]
            p += self._m[k]

net = ValueNet()
print(f"ValueNet parameter count: {net.n_params()} "
      f"(order-of-magnitude match to the ~7.5K budget in the plan)")


In [ ]:
# Phase B / B1c: training loop with early stopping on VALIDATION WIN-RATE
# (not validation loss -- the plan is explicit that loss and win-rate can diverge
# on a small, correlated self-play dataset).

def synthesize_smoke_test_data(n_games=40, decisions_per_game=12, seed=0):
    """Synthetic (state, outcome) pairs so the training loop is exercised even
    before real self-play logs exist. Replace with `train_records`/`val_records`
    once Phase A logging is live."""
    rng = np.random.default_rng(seed)
    X, y, gid = [], [], []
    for g in range(n_games):
        outcome = float(rng.integers(0, 2))
        for _ in range(decisions_per_game):
            X.append(rng.normal(outcome - 0.5, 0.3, size=STATE_DIM).astype(np.float32))
            y.append(outcome)
            gid.append(g)
    return np.array(X), np.array(y, dtype=np.float32), np.array(gid)

X, y, gid = synthesize_smoke_test_data()
val_mask = gid >= int(0.8 * gid.max())
X_train, y_train = X[~val_mask], y[~val_mask]
X_val, y_val = X[val_mask], y[val_mask]

def validation_win_rate(net, X_val, y_val, threshold=0.5):
    yhat, _ = net.forward(X_val)
    pred = (yhat.ravel() >= threshold).astype(np.float32)
    return float((pred == y_val).mean())

def train(net, X_train, y_train, X_val, y_val,
          lr=5e-3, momentum=0.9, l2=1e-4, batch_size=32,
          max_epochs=200, patience=15, seed=0):
    rng = np.random.default_rng(seed)
    best_val = -1.0
    best_params = None
    epochs_no_improve = 0
    n = X_train.shape[0]

    for epoch in range(max_epochs):
        order = rng.permutation(n)
        for start in range(0, n, batch_size):
            idx = order[start:start + batch_size]
            yhat, cache = net.forward(X_train[idx])
            grads = net.backward(cache, y_train[idx], l2=l2)
            net.step(grads, lr=lr, momentum=momentum)

        val_wr = validation_win_rate(net, X_val, y_val)
        if val_wr > best_val:
            best_val = val_wr
            best_params = {k: v.copy() for k, v in net.params().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                break

    if best_params is not None:
        for k, v in net.params().items():
            v[...] = best_params[k]
    return best_val, epoch + 1

best_val_wr, epochs_run = train(net, X_train, y_train, X_val, y_val)
print(f"best validation win-rate: {best_val_wr:.3f} after {epochs_run} epochs "
      f"(smoke test on synthetic data -- rerun against real game_log.jsonl once populated)")


### Wiring — value network as a tie-breaker

The value network does **not** replace `score_option()`. It only resolves near-ties: if
the top-scoring heuristic options are within a configurable margin (**±500 points**), the
value network's win-probability estimate for the resulting state breaks the tie.


In [ ]:
# Phase B / B1d: tie-breaker wiring

TIE_MARGIN = 500

def resolve_with_value_net(scored_options, net, state_of_option_fn):
    """`scored_options`: list of (heuristic_score, option) from score_option().
    `state_of_option_fn`: option -> resulting state vector (np.ndarray[84]).
    Only breaks ties among options within TIE_MARGIN of the top heuristic score --
    the symbolic engine still makes every decision that isn't ambiguous.
    """
    if not scored_options:
        return None
    scored_options = sorted(scored_options, key=lambda t: t[0], reverse=True)
    top_score = scored_options[0][0]
    contenders = [opt for score, opt in scored_options if top_score - score <= TIE_MARGIN]

    if len(contenders) == 1:
        return contenders[0]

    states = np.stack([state_of_option_fn(opt) for opt in contenders])
    win_probs, _ = net.forward(states)
    best_idx = int(np.argmax(win_probs.ravel()))
    return contenders[best_idx]

# smoke test with dummy options
dummy_options = [(9000, "evolve_active"), (8950, "evolve_bench"), (6500, "play_supporter")]
chosen = resolve_with_value_net(dummy_options, net, lambda opt: build_state_vector(obs))
print("tie-breaker chose:", chosen)


### B2. One-ply lookahead (if time remains)

Replace the greedy lethal-check with **one-ply lookahead using the engine's own native
`search_begin`/`search_step` substrate** — not hand-rolled MCTS. The engine's search
primitives are already correctness-tested by the competition harness; a hand-rolled MCTS
on a 3-week clock is much more likely to introduce a state-mutation bug than to out-search
the native implementation at depth 1.


In [ ]:
# Phase B / B2: one-ply lookahead scaffold (native search substrate)

def one_ply_lookahead(state, legal_options, evaluate_terminal_fn):
    """Wraps the engine-native search_begin/search_step calls. Falls back to a
    plain one-step simulation when the real `cg` package isn't importable, so
    this cell is exercisable outside the Kaggle harness."""
    if USE_MOCK_ENGINE:
        # Mock: score each option by a trivial 1-step lookahead heuristic only,
        # to validate the control flow -- NOT a substitute for the real engine.
        return max(legal_options, key=lambda opt: evaluate_terminal_fn(state, opt))

    # --- Real harness path ---
    # handle = cg.search_begin(state)
    # results = []
    # for opt in legal_options:
    #     outcome_state = cg.search_step(handle, opt)
    #     results.append((opt, evaluate_terminal_fn(outcome_state, None)))
    # cg.search_end(handle)
    # return max(results, key=lambda t: t[1])[0]
    raise NotImplementedError("wire to real cg.search_begin/search_step on the Kaggle harness")

def dummy_evaluate(state, opt):
    return hash((str(state)[:8], opt)) % 100  # placeholder terminal evaluation

print(one_ply_lookahead("state", ["attack_a", "attack_b"], dummy_evaluate))


## Phase C: Stabilize, Don't Innovate (Aug 9 → Aug 16)

### Non-Regression Gate (mandatory before every submission)

- Run **N ≥ 30** matches against the previous best submission
- Report win-rate with a **Wilson score interval**, not a raw percentage — Wilson (1927),
  *"Probable Inference, the Law of Succession, and Statistical Inference,"* JASA
- Promotion threshold: **≥55% win-rate**, matching AlphaGo Zero's own gating rule
  (Silver et al. 2017, *"Mastering the game of Go without human knowledge,"* Nature)
- A raw 60% at n=30 has a genuinely wide confidence band — don't trust it without the interval

Lock final submission by **Aug 16 23:59 UTC**.


In [ ]:
# Phase C: Wilson score interval + non-regression gate

def wilson_interval(wins: int, n: int, z: float = 1.96):
    """95% Wilson score interval for a binomial proportion (Wilson, 1927).
    More reliable than a normal (Wald) interval at small n / extreme proportions,
    which is exactly the n>=30 regime this gate operates in.
    """
    if n == 0:
        return 0.0, 0.0, 0.0
    p_hat = wins / n
    denom = 1 + z**2 / n
    center = p_hat + z**2 / (2 * n)
    margin = z * math.sqrt(p_hat * (1 - p_hat) / n + z**2 / (4 * n**2))
    low = (center - margin) / denom
    high = (center + margin) / denom
    return p_hat, low, high


def non_regression_gate(wins: int, n: int, threshold: float = 0.55, min_n: int = 30):
    """Promote only if:
      (a) at least `min_n` matches were played, and
      (b) the Wilson lower bound clears the threshold -- i.e. we're confident the
          *true* win-rate is >= threshold, not just the observed sample mean.
    """
    if n < min_n:
        return False, f"only {n} matches played (need >= {min_n})"
    p_hat, low, high = wilson_interval(wins, n)
    verdict = low >= threshold
    msg = (f"win-rate={p_hat:.3f} (95% CI [{low:.3f}, {high:.3f}]), "
           f"threshold={threshold} -> {'PROMOTE' if verdict else 'HOLD'}")
    return verdict, msg

# Example: the plan's own cautionary case -- a raw 60% at n=30
promote, msg = non_regression_gate(wins=18, n=30)  # 18/30 = 60%
print("raw 60% @ n=30:", msg)

# A cleaner pass at higher n
promote2, msg2 = non_regression_gate(wins=40, n=60)  # ~66.7%
print("~67% @ n=60:   ", msg2)


## Report Phase (Aug 17 → Sept 13)

Games keep running passively through Aug 31. Job shifts entirely to the **Strategy
report** (2,000 words, aligned to judging criteria):

1. **Agent Architecture** — Neuro-symbolic hybrid: symbolic rule engine makes every
   decision today, neural value network is structurally present, trained on self-play
   data, wired as tie-breaker. Honest framing, not inflated.
2. **Deck Construction Rationale** — Why Mega Lucario ex, matchup spread analysis,
   card-by-card justification (20% of Strategy grade).
3. **Decision Explainability** — Trace specific game states through `score_option()`,
   show how the scoring hierarchy resolves ambiguous situations.
4. **Ablation Data** — Performance with/without value network tie-breaker, with/without
   specific heuristic rules. This is the data Phase B produces.
5. **Research Framing** — four honest novelty claims, each independently checkable
   against this notebook's code, not against outside citations:
   - (a) Neuro-symbolic architecture stated honestly ("symbolic decides, neural is
     present but inert until trained")
   - (b) Using engine-native `search_begin`/`search_step` instead of hand-rolled MCTS
   - (c) Adversarially-verified parser construction (cross-checking enums/fields against
     a real captured observation, per Phase A / Task 1 — not assumed)
   - (d) Single-archetype specialist hypothesis ("depth over breadth wins more Elo per
     engineering-hour on a 3-week clock")


In [ ]:
# Report Phase: ablation harness skeleton -- produces the numbers claim (4) needs

def run_ablation(agent_variants: dict, n_matches_per_variant: int = 30):
    """`agent_variants`: name -> callable that plays one match and returns 1/0/0.5.
    Reports Wilson intervals per variant so the report can cite real, computed
    confidence bounds instead of a bare win/loss count."""
    results = {}
    for name, play_match_fn in agent_variants.items():
        wins = sum(1 for _ in range(n_matches_per_variant) if play_match_fn() == 1)
        p_hat, low, high = wilson_interval(wins, n_matches_per_variant)
        results[name] = {"win_rate": p_hat, "ci_low": low, "ci_high": high,
                          "n": n_matches_per_variant}
    return results

def _mock_match_with_tiebreaker():
    return random.random() < 0.58  # placeholder win probability

def _mock_match_heuristic_only():
    return random.random() < 0.50

ablation_results = run_ablation({
    "heuristic_only": _mock_match_heuristic_only,
    "heuristic_plus_value_tiebreaker": _mock_match_with_tiebreaker,
})
for name, r in ablation_results.items():
    print(f"{name:35s} win_rate={r['win_rate']:.3f} CI=[{r['ci_low']:.3f}, {r['ci_high']:.3f}] n={r['n']}")


## Scoring System Clarification

> Two different scoring systems exist. Do not conflate them.

| System | What It Is | Units | Visible To |
|--------|-----------|-------|------------|
| `score_option()` internal | Arbitrary integer ranking of legal options within one decision | Points (50000, 10000, ...) | Only our agent |
| Competition skill rating | TrueSkill/Elo Bayesian rating from matchmaking pool | Rating (starts at 600) | Kaggle leaderboard |

The sample rule-based agent (same Mega Lucario ex deck) sits at **664.2**. Daily median
climbed from ~628 to ~1,180 over 38 days. The target is **"beat a rising median by a
statistically real margin"** (i.e., clear the Wilson lower bound above), not a static
number.


## Process Fixes

Losing 2 of 3 teamwork runs to a server restart is a **checkpointing failure in our
workflow**, not an infrastructure problem outside our control. Before launching any more
multi-hour agent runs:

1. Ensure the working `main.py` is committed/backed up before any teamwork run starts
2. Teamwork runs should write incremental outputs (not one final dump)
3. Keep the current working submission as a separate, untouched copy


In [ ]:
# Process fix: pre-run backup helper

def backup_before_teamwork_run(main_py_path="main.py", backup_dir="backups"):
    """Call this as the very first step of any multi-hour agent run. Cheap
    insurance against exactly the failure mode that cost 2 of 3 prior runs."""
    src = Path(main_py_path)
    backup_root = Path(backup_dir)
    backup_root.mkdir(exist_ok=True)
    if not src.exists():
        print(f"WARNING: {main_py_path} not found -- nothing to back up")
        return None
    stamp = time.strftime("%Y%m%d_%H%M%S")
    dst = backup_root / f"main_{stamp}.py"
    shutil.copy2(src, dst)
    print(f"backed up {src} -> {dst}")
    return dst

# backup_before_teamwork_run()  # uncomment before starting a real teamwork run
print("backup helper defined -- call backup_before_teamwork_run() before every teamwork run")


## Immediate Next Step

**Phase A, Task 1**: Capture and inspect a real game observation. This is the single
highest-leverage unclosed task — everything in Phase B depends on knowing the actual wire
format, not our assumptions about it.

Run the `capture_and_inspect` cell above against a **real** observation (not
`mock_observation()`) before touching anything else in this notebook.

---
*This plan is honest about what we don't know and conservative about timelines. The
ceiling is real; breaking it requires the neural components, which require the training
data, which require the verification loop to close first.*
